In [ ]:
import numpy as np
from scipy import signal
import tensorflow as tf
from tensorflow.keras import layers, Model, initializers, Sequential
from optic.models.devices import mzm, photodiode, edfa, iqm, coherentReceiver, pdmCoherentReceiver, basicLaserModel
from optic.models.channels import linearFiberChannel, ssfm
from optic.comm.modulation import modulateGray, grayMapping
from optic.comm.sources import bitSource, symbolSource
from optic.dsp.core import upsample, pulseShape, pnorm, anorm, signalPower, firFilter, decimate, symbolSync,phaseNoise

try:
    from optic.dsp.coreGPU import checkGPU
    if checkGPU():
        from optic.dsp.coreGPU import firFilter
    else:
        from optic.dsp.core import firFilter
except ImportError:
    from optic.dsp.core import firFilter

from optic.utils import parameters, dBm2W, ber2Qfactor
from optic.plot import eyediagram, pconst, plotPSD
import matplotlib.pyplot as plt
from scipy.special import erfc
from tqdm.notebook import tqdm
import scipy as sp
import scipy.constants as const

try:
    from optic.models.modelsGPU import manakovSSF
except:
    from optic.models.channels import manakovSSF

from optic.dsp.equalization import edc, mimoAdaptEqualizer, ffe
from optic.dsp.carrierRecovery import cpr
from optic.comm.metrics import fastBERcalc, monteCarloGMI, monteCarloMI, calcEVM, bert
from optic.dsp.clockRecovery import gardnerClockRecovery


import logging as logg
logg.basicConfig(level=logg.INFO, format='%(message)s', force=True)
import time

In [ ]:
from IPython.core.display import HTML
from IPython.core.pylabtools import figsize

HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

In [ ]:
# --------------------------------------------------------------------------------------------------------------------------------------
# Power Amplifier (PA) Functions
# ---------------------------------------------------------------------------------------------------------------------------------------

def rapp_pa(x, Vsat, p=2.0):
    """
    Memoryless Rapp PA model for complex baseband input.

    Parameters
    ----------
    x : np.ndarray
        Complex input waveform in volts.
    Vsat : float
        Saturation voltage/amplitude.
    p : float
        Rapp smoothness exponent.

    Returns
    -------
    y : np.ndarray
        Output after memoryless PA nonlinearity.
    """
    mag = np.abs(x)
    gain = 1.0 / (1.0 + (mag / Vsat) ** (2 * p)) ** (1.0 / (2 * p))
    return x * gain


def butter_lpf_complex(x, Fs, f3dB, order=3):
    """
    Apply an order-N Butterworth LPF to a complex baseband waveform.
    """
    wn = f3dB / (Fs / 2)

    if wn >= 1.0:
        raise ValueError(
            f"Butterworth cutoff must be below Nyquist. Got f3dB={f3dB/1e9:.2f} GHz, "
            f"Fs/2={(Fs/2)/1e9:.2f} GHz."
        )

    b, a = signal.butter(order, wn, btype='low')

    y_i = signal.lfilter(b, a, np.real(x))
    y_q = signal.lfilter(b, a, np.imag(x))

    return y_i + 1j * y_q


def pa_model_physical(x, Fs, gain_linear, BLPF_enable, BO_dB=5.0, p=2.0, f3dB=10e9, order=3):
    """
    Physical PA model:
        x -> linear gain -> Rapp compression -> Butterworth LPF

    Parameters
    ----------
    x : np.ndarray
        Complex baseband input waveform (dimensionless DSP waveform).
    Fs : float
        Sample rate [Hz].
    gain_linear : float
        Small-signal linear voltage gain. This sets the nominal IQM drive level.
    BO_dB : float
        Back-off in dB, used to set Vsat relative to the RMS value AFTER gain.
    p : float
        Rapp exponent.
    f3dB : float
        PA 3-dB bandwidth [Hz].
    order : int
        Butterworth filter order.

    Returns
    -------
    y : np.ndarray
        PA output waveform in volts, ready to drive the IQM.
    info : dict
        Diagnostic information.
    """

    # 1) Linear gain stage -> now waveform is in volts
    x_amp = gain_linear * x

    # 2) Compute RMS after gain
    Vrms_in = np.sqrt(np.mean(np.abs(x_amp) ** 2))

    # 3) Saturation voltage from back-off
    Vsat = Vrms_in * 10 ** (BO_dB / 20)

    # 4) Nonlinear compression
    y_nl = rapp_pa(x_amp, Vsat=Vsat, p=p)

    # 5) PA bandwidth limitation
    if BLPF_enable:
        y = butter_lpf_complex(y_nl, Fs=Fs, f3dB=f3dB, order=order)
    else:
        y = y_nl

    info = {
        "gain_linear": gain_linear,
        "Vrms_in_V": Vrms_in,
        "Vsat_V": Vsat,
        "BO_dB": BO_dB,
        "p": p,
        "f3dB_Hz": f3dB,
        "peak_out_V": np.max(np.abs(y)),
        "rms_out_V": np.sqrt(np.mean(np.abs(y) ** 2)),
    }

    return y, info

In [ ]:
def select_cpr_mode(paramCPR_, CPR_Mode, Ts, M):
    paramCPR = paramCPR_
    paramCPR.Ts = Ts
    paramCPR.M  = M
    paramCPR.returnPhases = True

    if CPR_Mode.lower() == "bps":
        paramCPR.alg = "bps"
        if M == 16:
            paramCPR.N = 25
            paramCPR.B = 64
        elif M == 32:
            paramCPR.N = 31
            paramCPR.B = 128
        elif M == 64:
            #paramCPR.N = 81
            paramCPR.N = 101
            paramCPR.B = 2048
            #paramCPR.B = 1024
        elif M == 256:
            paramCPR.N = 81
            paramCPR.B = 512

    elif CPR_Mode.lower() == "ddpll":
        paramCPR.alg = "ddpll"
        if M == 16:
            # recommended DDPLL parameters
            paramCPR.tau1 = 1/(2*np.pi*10e3)
            paramCPR.tau2 = 1/(2*np.pi*10e3)
            paramCPR.Kv   = 0.1
        elif M == 32:
            paramCPR.tau1 = 1/(2*np.pi*20e3)
            paramCPR.tau2 = 1/(2*np.pi*20e3)
            paramCPR.Kv   = 0.1
            
        elif M == 64:
            #paramCPR.tau1 = 1/(2*np.pi*30e3)
            #paramCPR.tau2 = 1/(2*np.pi*30e3)
            #paramCPR.Kv   = 0.15
            paramCPR.Kv = 0.07
            paramCPR.tau1 = 1/(2*np.pi*5e6)
            paramCPR.tau2 = 1/(2*np.pi*5e6)
        elif M == 256:
            paramCPR.tau1 = 1/(2*np.pi*40e3)
            paramCPR.tau2 = 1/(2*np.pi*40e3)
            paramCPR.Kv   = 0.12  # faster tracking needed
          
    else:
        raise ValueError("CPR_Mode must be 'bps' or 'ddpll'")

    return paramCPR

In [ ]:
def select_equalizer_mode(M, Data_Aided, paramEq_):
    """
    Selects the equalizer algorithm and step sizes 
    based on M, Data_Aided flag, and chosen mode.
    """
    paramEq = paramEq_
    if Data_Aided:
        if M == 4:
            # For QPSK
            paramEq.alg = ['cma', 'cma']
            paramEq.mu  = [5e-3, 1e-3]
            paramEq.numIter = 2
        elif M == 16:
            # For 16-QAM
            paramEq.alg = ['da-rde', 'rde']
            paramEq.mu  = [5e-3, 5e-4]
            paramEq.numIter = 3
        elif M == 32:
            paramEq.alg = ['da-rde', 'rde']
            paramEq.mu  = [2e-3, 5e-4]
            paramEq.numIter = 4
            paramEq.nTaps = 45
        elif M == 64:
            paramEq.alg = ['da-rde', 'rde']
            paramEq.mu  = [8e-4, 4e-4]
            paramEq.mu  = [8e-4, 3e-4]
            paramEq.numIter = 6
            paramEq.nTaps = 65
            #paramEq.alg = ['da-rde', 'cma']
            #paramEq.mu = [3e-4, 5e-5]
            #paramEq.numIter = 4
            #paramEq.nTaps = 85
        elif M == 256:
            paramEq.alg = ['da-rde', 'da-rde']
            paramEq.mu  = [2e-3, 5e-4]
            paramEq.numIter = 6
            paramEq.nTaps = 180
            
            
    else:  # Blind mode
        if M == 4:
            paramEq.alg = ['cma','cma']
            paramEq.mu  = [5e-3, 1e-3]
            paramEq.numIter = 2
        elif M == 16:
            paramEq.alg = ['cma','rde']
            paramEq.mu  = [5e-3, 1e-3]
            paramEq.numIter = 3
        elif M == 32:
            paramEq.alg = ['cma','rde']
            paramEq.mu  = [2e-3, 5e-4]
            paramEq.numIter = 4
            paramEq.nTaps = 45
        elif M == 64:
            paramEq.alg = ['cma', 'rde']
            paramEq.mu  = [1e-3, 1e-3]
            paramEq.numIter = 5
            paramEq.nTaps = 55
        elif M == 256:
            paramEq.alg = ['cma', 'rde']
            paramEq.mu  = [5e-4, 5e-4]
            paramEq.numIter = 6
            paramEq.nTaps = 65
   
            
    return paramEq

In [ ]:
def perf_calc(symbTx, y_CPR_1, d_, M, paramSymb):
    """Performance Metric Exploration for Single Polarization"""
    d = d_
    discard = 5000
    ind = np.arange(discard, len(symbTx) - discard)
    
    # Remove phase ambiguity for all M (optional: for QAM)
    if M in [4, 16, 32, 64, 128]:
        d = symbTx  # or processed reference symbols

    # Compute metrics
    BER, SER, SNR = fastBERcalc(y_CPR_1[ind], d[ind], M, 'qam', px=paramSymb.px)
    EVM = calcEVM(y_CPR_1[ind], M, 'qam', d[ind])
    Qfactor = ber2Qfactor(BER[0])

    print(' SER: %.3e,  '%(SER[0]))
    print(' BER: %.3e   '%(BER[0]))
    print(' SNR: %.3f dB'%(SNR[0]))
    print(' EVM: %.3f %%'%(EVM[0]*100))
    print(' Qfactor: %.3f,  '%(Qfactor))

    return BER[0], SER[0], SNR[0], EVM[0], Qfactor

In [ ]:
# ----------------------------------------------
# Simulation of the Optical System (parametric version)
# -----------------------------------------------

def simulate_optical_system(
    symbTx,
    no_symbols_sent,
    M,
    PA_enable=True,
    Data_Aided=True,
    SpS=16,
    SpSout=2,
    Fs=None,
    mzmScale=0.5,
    Vpi=2,
    BLPF_enable=True,
    PA_BO_dB=3,
    PA_p=2.0,
    PA_f3dB=18.5e9,
    P_launch_dBm=0,
    Rs = 32e9,                
    rollOff = 0.01,           
    nFilterTaps = 1024,       
    laserLinewidth = 100e3, 
    FO  = -128e6,
    CPR_Mode = "bps",
    pulse_type="rrc",
    ch_Ltotal_km=80,
    ch_Lspan_km=80,
    ch_alpha_dB_per_km=0.2,
    ch_D_ps_nm_km=16,
    ch_gamma=1.3,
    ch_Fc=193.1e12,
    ch_hz_km=0.5,
    ch_prgsBar=True,
    ch_amp="edfa",
    ch_NF_dB=4.5,
    lo_P_dBm=2,
    lo_RIN_var=0,
    lo_freq_shift_base_hz=0,
    pn_tx_seed=123,
    lo_rx_seed=789,
    pd_seed=1011,
    pd_ideal=True,
    edc_Fs=None,
    pa_order=3,
    ):                
    
    # (derived) params
    if Fs is None:
        Fs = Rs * SpS
    # 3) UpSampling and FIR Parameters
    paramPulse = parameters()
    paramPulse.pulseType = pulse_type
    paramPulse.nFilterTaps = nFilterTaps
    paramPulse.rollOff = rollOff
    paramPulse.SpS = SpS

    # 4) IQM Parameters
    paramIQM = parameters()
    paramIQM.Vpi = Vpi
    paramIQM.VbI = -Vpi
    paramIQM.VbQ = -Vpi
    paramIQM.Vphi = Vpi/2

    # 5) Optical Carrier / LO field (Ein)
    sigTx_length = no_symbols_sent * SpS
    if laserLinewidth and laserLinewidth > 0:
        phi_pn = phaseNoise(laserLinewidth, sigTx_length, 1 / Fs, seed=pn_tx_seed)
        sigLO = np.exp(1j * phi_pn)
    else:
        sigLO = np.ones_like(sigTx_length, dtype=complex)


    # -----------------------------------------------
    # Channel Parameters
    #------------------------------------------------

    # 1) Optical Channel Parameters
    paramCh = parameters()
    paramCh.Ltotal = ch_Ltotal_km
    paramCh.Lspan = ch_Lspan_km
    paramCh.alpha = ch_alpha_dB_per_km
    paramCh.D = ch_D_ps_nm_km
    paramCh.gamma = ch_gamma
    paramCh.Fc = ch_Fc
    paramCh.hz = ch_hz_km
    paramCh.prgsBar = ch_prgsBar
    paramCh.Fs = Fs
    paramCh.amp = ch_amp
    paramCh.NF = ch_NF_dB
    #paramCh.seed = 456


    # -----------------------------------------------
    # Receiver Parameters
    #------------------------------------------------

    # 1) local oscillator (LO) parameters:

    paramLO = parameters()
    paramLO.P = lo_P_dBm
    paramLO.lw = laserLinewidth
    paramLO.RIN_var = lo_RIN_var
    paramLO.Fs = Fs
    paramLO.seed = lo_rx_seed
    paramLO.freqShift = lo_freq_shift_base_hz + FO

    # 2) Front-End Parameters and photodiode paramters

    # Frontend parameters
    paramFE = parameters()
    paramFE.Fs = Fs

    # Photodiodes parameters
    paramPD = parameters()
    paramPD.B = Rs
    paramPD.Fs = Fs
    paramPD.ideal = pd_ideal
    paramPD.seed = pd_seed

    # 3) Pulseshaping in the reciever using rrc filter
    paramRxPulse = parameters()
    paramRxPulse.SpS = SpS
    paramRxPulse.nFilterTaps = nFilterTaps
    paramRxPulse.rollOff = rollOff
    paramRxPulse.pulseType = pulse_type

    # 4) Decimation Parameters
    paramDec = parameters()
    paramDec.SpSin  = SpS
    paramDec.SpSout = SpSout

    # 5) Chromatic Dispersion Parameters
    paramEDC = parameters()
    paramEDC.L = paramCh.Ltotal
    paramEDC.D = paramCh.D
    paramEDC.Fc = paramCh.Fc
    paramEDC.Rs = Rs
    paramEDC.Fs = 2 * Rs if edc_Fs is None else edc_Fs

    # 6) Adaptive Equalization Parameters
    paramEq = parameters()
    paramEq.nTaps = 35
    paramEq.SpS = paramDec.SpSout
    paramEq.numIter = 2
    paramEq.storeCoeff = False
    paramEq.M = M
    paramEq.shapingFactor = 0
    paramEq.constType = "qam"
    paramEq.prgsBar = False

    # 7) Data-Aided or Blind Reciever Equalization
    # Can be set here or not
    #Data_Aided = True

    # 8) Carrier and Phase recovery parameters using bps
    paramCPR = parameters()
    paramCPR.alg = 'bps'
    paramCPR.M   = M
    paramCPR.constType ="qam"
    paramCPR.shapingFactor = 0
    paramCPR.N   = 25
    paramCPR.B   = 64
    paramCPR.returnPhases = True
    paramCPR.Ts = 1/Rs




    # -----------------------------------------------------------------------------------------------------------------------------------------------------
    # TRANSMITTER

    # 2) Upsampling + pulse shaping
    pulse = pulseShape(paramPulse)
    symbolsUp = upsample(symbTx, SpS)
    sigTx = firFilter(pulse, symbolsUp)

    # 3) Choose nominal small-signal PA gain so the nominal drive is around mzmScale * Vpi
    target_peak_V = mzmScale * Vpi
    peak_sigTx = np.max(np.abs(sigTx))

    if peak_sigTx == 0:
        raise ValueError("sigTx peak is zero; cannot set PA gain.")

    gain_linear = target_peak_V / peak_sigTx

    # 4) Driver amplifier / PA output directly in volts
    if PA_enable:
        u_drive, paInfo = pa_model_physical(
            sigTx,
            Fs=Fs,
            gain_linear=gain_linear,
            BLPF_enable=BLPF_enable,
            BO_dB=PA_BO_dB,
            p=PA_p,
            f3dB=PA_f3dB,
            order=pa_order,
        )
    else:
        u_drive = gain_linear * sigTx
        paInfo = {
            "gain_linear": gain_linear,
            "Vrms_in_V": np.sqrt(np.mean(np.abs(u_drive) ** 2)),
            "Vsat_V": None,
            "BO_dB": None,
            "p": None,
            "f3dB_Hz": None,
            "peak_out_V": np.max(np.abs(u_drive)),
            "rms_out_V": np.sqrt(np.mean(np.abs(u_drive) ** 2)),
        }

    # 5) IQ modulation: PA output drives the IQM directly
    sigTxo = iqm(sigLO, u_drive, paramIQM)

    # 6) Set launched optical power
    P_launch_W = dBm2W(P_launch_dBm)
    sigTxo = np.sqrt(P_launch_W) * pnorm(sigTxo)

    # End of Transmitter
    # -----------------------------------------------------------------------------------------------------------------------------------------------------

    # -----------------------------------------------------------------------------------------------------------------------------------------------------
    # CHANNEL

    sigCh = ssfm(sigTxo, paramCh)

    # End of CHANNEL
    # -----------------------------------------------------------------------------------------------------------------------------------------------------

    # -----------------------------------------------------------------------------------------------------------------------------------------------------
    # RECEIVER

    # 1) Generate CW laser LO field
    paramLO.Ns = len(sigCh)
    sigLO_Rx = basicLaserModel(paramLO)

    # 2) Coherent receiver for single-polarization
    sigRxFrontEnd = coherentReceiver(sigCh, sigLO_Rx, paramFE, paramPD)

    # 3) Pulse shaping
    pulse = pulseShape(paramRxPulse)
    sigRxPulseShape = firFilter(pulse, sigRxFrontEnd)

    # 4) Decimation
    sigRxDecimation = decimate(sigRxPulseShape, paramDec)

    # 5) Chromatic Dispersion Compensation
    sigRxCD = edc(sigRxDecimation, paramEDC)

    # 6) Symbol Synchronization with the SymbTx
    symbRxCD = symbolSync(sigRxCD, symbTx, 2)

    # 7) Power Normalization
    x = pnorm(sigRxCD)
    d = pnorm(symbRxCD)

    if M==256 and Data_Aided:
        paramEq.L = [int(0.5*d.shape[0]), int(0.5*d.shape[0])]
    else:
        paramEq.L = [int(0.2*d.shape[0]), int(0.8*d.shape[0])]
   
    #paramEq.L         = [int(0.8 * d.shape[0])]    # or d.shape[0] - 20k
    # ------------------------------------------------
    # 5) EQUALIZATION (via DSP SWITCH)
    # ------------------------------------------------
    paramEq = select_equalizer_mode(M, Data_Aided, paramEq)

    if Data_Aided:
        print(" adied")
        y_EQ = mimoAdaptEqualizer(x, paramEq, d)
    else:
        print("no adied")
        y_EQ = mimoAdaptEqualizer(x, paramEq, None)

    #y_EQ, h_rls = rls_single_pol(x, d, L=21, lam=0.995, delta=1e3)

    # ------------------------------------------------
    # 6) Frequency Offset Compensation
    # ------------------------------------------------
    Ts = 1 / Rs
    paramCPR = select_cpr_mode(paramCPR, CPR_Mode, Ts, M)
    if CPR_Mode == "ddpll":
        print("no bps")
        
        y_EQ_2D = y_EQ.reshape(-1,1) if y_EQ.ndim == 1 else y_EQ
        
        symbTx_2D = symbTx.reshape(-1,1)
        y_CPR_1, phaseEst = cpr(y_EQ_2D, param=paramCPR, symbTx=symbTx_2D)
        y_CPR_1= y_CPR_1.flatten()
    else:
        print("yes bps")
        y_CPR_1, phaseEst = cpr(y_EQ, paramCPR)

    return y_CPR_1, d, phaseEst


In [ ]:
def intialise_paramSymb(M, nBits, seed=444):
    # Symbol generation    
    paramSymb = parameters()
    paramSymb.nSymbols = int(nBits // np.log2(M))  # symbols = bits / log2(M)
    paramSymb.M = M
    paramSymb.constType = "qam"                    # 'qam' with M=4 -> QPSK
    paramSymb.dist = "uniform"                     # uniform symbol probabilities
    paramSymb.seed = 444
    paramSymb.shapingFactor = 0

    constSymb = grayMapping(paramSymb.M, paramSymb.constType)
    if paramSymb.dist == "uniform":
        px = np.ones(paramSymb.M) / paramSymb.M
    elif paramSymb.probDist == "maxwell-boltzmann":
        px = np.exp(-paramSymb.shapingFactor * np.abs(constSymb) ** 2)
        px = px / np.sum(px)
    else:
        raise ValueError("Invalid probability distribution.")
    paramSymb.px = px
    return paramSymb

## System First Trial - No DPD

In [ ]:
# M = 16
# nBits = 400000
# SpSout = 2
# paramSymb = intialise_paramSymb(M, nBits)


In [ ]:
# symbTx = symbolSource(paramSymb)


# y_CPR_1, d, phaseEst = simulate_optical_system(symbTx, len(symbTx), M, PA_enable=True, Data_Aided=True, SpSout=SpSout, mzmScale=0.8)


# perf_calc(symbTx, y_CPR_1, d, M, paramSymb)


# discard = 5000
# # plot constellations
# pconst(y_CPR_1[discard:-discard])

# # plotting eye diagrams of sigTx
# eyediagram(y_CPR_1.real[discard:-discard], y_CPR_1.real.size-2*discard, SpSout, plotlabel='signal at Rx REAL', ptype='fancy')
# eyediagram(y_CPR_1.imag[discard:-discard], y_CPR_1.imag.size-2*discard, SpSout, plotlabel='signal at Rx IMAGNIARY', ptype='fancy')


## System Benchmark (No DPD)

In [ ]:
# results = []

# # modulation orders to test
# mod_orders = [16,64,256]

# # DA and CPR modes
# modes_DataAided = [True, False]
# modes_CPR = [ "bps","ddpll"]

# for M_test in mod_orders:
#     M = M_test
#     paramSymb = intialise_paramSymb(M, nBits) # need to update it everytime M changes
#     for da in modes_DataAided:
#         for cpr_mode in modes_CPR:
#             print("\n===================================================")
#             print(f" Running:  M={M}, Data_Aided={da}, CPR_Mode={cpr_mode}")
#             print("===================================================\n")
        
#             # fresh symbol stream each run
#             symbTx = symbolSource(paramSymb)
            
#             # run sys
#             y_CPR_1, d, phaseEst = simulate_optical_system(symbTx, len(symbTx), M, 
#                                                            Data_Aided=da, 
#                                                            SpSout=SpSout,
#                                                            CPR_Mode=cpr_mode)

#             discard = 5000

#             # optional: plot constellations
#             pconst(y_CPR_1[discard:-discard])

#             # plotting eye diagrams
#             eyediagram(y_CPR_1.real[discard:-discard],
#                         y_CPR_1.real.size-2*discard,
#                         SpSout,
#                         plotlabel=f'signal at Rx REAL, M={M}', ptype='fancy')
#             eyediagram(y_CPR_1.imag[discard:-discard],
#                         y_CPR_1.imag.size-2*discard,
#                         SpSout,
#                         plotlabel=f'signal at Rx IMAGINARY, M={M}', ptype='fancy')


#             # performance
#             BER, SER, SNR, EVM, Q = perf_calc(symbTx, y_CPR_1, d, M, paramSymb)


#             results.append({
#                 "Modulation": M,
#                 "Data_Aided": da,
#                 "CPR": cpr_mode,
#                 "BER": BER,
#                 "SER": SER,
#                 "SNR": SNR,
#                 "EVM": EVM,
#                 "Qfactor": Q
#             })


In [ ]:
# print("\n==================== SUMMARY TABLE ====================\n")
# for r in results:
#     print(f"DA={r['Data_Aided']}, CPR={r['CPR']}: "
#           f"BER={r['BER']:.2e}, SER={r['SER']:.2e}, "
#           f"SNR={r['SNR']:.2f} dB, EVM={r['EVM']*100:.1f} %, Q={r['Qfactor']:.2f}")

# DPD

In [ ]:
def build_model():
    inputs = layers.Input(shape=(None, 2)) # 2 for I and Q

    ## approach 1:
    sec_a = layers.Conv1D(2, 101, padding='same')(inputs) # 100 taps was a sweet spot, 20-ish fails to converge, tiker with different values.


    ## approach 2:
    # sec_a = layers.DepthwiseConv1D(100, padding='same', use_bias=False)(inputs) # 100 taps was a sweet spot, 20-ish fails to converge, tiker with different values.

    ## approach 3:
    # input_i = layers.Lambda(lambda x: x[:, :, 0:1])(inputs)
    # input_q = layers.Lambda(lambda x: x[:, :, 1:2])(inputs)
    # sec_a_i = layers.Conv1D(1, 100, padding='causal', use_bias=False, name='fir_i')(input_i)
    # sec_a_q = layers.Conv1D(1, 100, padding='causal', use_bias=False, name='fir_q')(input_q)
    # sec_a = layers.Concatenate(axis=-1)([sec_a_i, sec_a_q])


    nonlinear_1 = layers.Dense(20, activation=tf.math.sin)(sec_a)
    nonlinear_2 = layers.Dense(20, activation=tf.math.sin)(nonlinear_1)
    nonlinear_3 = layers.Dense(2, activation='linear')(nonlinear_2)
    
    outputs = layers.Add()([sec_a, nonlinear_3]) 
    
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2), loss='mse') # ILA)
    return model


In [ ]:
def split_i_q(arr):
    return np.stack([np.real(arr), np.imag(arr)]).T

def merge_i_q(arr):
    if arr.ndim == 3:
        return arr[:,:,0] + 1j*arr[:,:,1]
    elif arr.ndim == 2:
        return arr[:,0] + 1j*arr[:,1]
    else:
        raise ValueError("Input array must be 2D or 3D.")

def preprocess(symbTx, seq_length=5000):
    reshabe_len = len(symbTx)//seq_length
    symbTx_nn = split_i_q(symbTx)
    symbTx_nn = symbTx_nn[:reshabe_len*seq_length].reshape(-1, seq_length, 2) # batching the symbols for training (shape: num_batches, seq_length, num_features)
    return symbTx_nn

def postprocess(symbTx_nn, original_symbTx, seq_length=5000):
    reconstructed_nn = symbTx_nn.reshape(-1, 2)
    reconstructed_complex = reconstructed_nn[:, 0] + 1j * reconstructed_nn[:, 1]
    
    total_len = len(original_symbTx)
    cutoff_point = (total_len // seq_length) * seq_length
    thrown_off_symbols = original_symbTx[cutoff_point:]
    
    return np.concatenate([reconstructed_complex, thrown_off_symbols])

In [ ]:
def train_DPD(M, nBits, iteration_cnt = 15, **kwargs):
    
    paramSymb = intialise_paramSymb(M, nBits, seed=333)
    symbTx = symbolSource(paramSymb)
    dpd_model = build_model()
    symbTx_nn = preprocess(symbTx)
    dpd_model.fit(symbTx_nn, symbTx_nn, epochs=500, verbose=0) # this line is important, = starting as a passthrough.
    best_ber = float('inf')

    for iteration in range(iteration_cnt):
        print(f"====== Iteration {iteration} ======")

        symbDPD = dpd_model.predict(symbTx_nn, verbose=0)
        x = merge_i_q(symbDPD).flatten()
        # x =   postprocess(symbDPD, symbTx)

        y_CPR_1, d, phaseEst = simulate_optical_system(x, len(x), M, **kwargs)


        ber, SER, SNR, EVM, Q = perf_calc(symbTx, y_CPR_1, d, M, paramSymb)
        if ber < best_ber:
            dpd_model.save_weights("best_model.weights.h5")
            best_ber = ber


        y_CPR_1_nn = preprocess(y_CPR_1)


        # y = (y_CPR_1 - np.mean(y_CPR_1)) / np.std(y_CPR_1)
        # y_nn = split_i_q(y).reshape(-1, seq_length, 2)

        dpd_model.fit(y_CPR_1_nn, symbDPD, epochs=100, verbose=0) # ILA


        ## PLEASE IGNORE THE BELOW CODE 
        # aux_model.fit(symbDPD, y_CPR_1_nn, epochs=200, verbose=0)
        # aux_model.trainable = False
        # dla_cascade.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
        
        # dla_cascade.fit(symbTx_nn, symbTx_nn, epochs=100, verbose=0)
        # aux_model.trainable = True
        # dla_cascade.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')


# Benchmarking W/ DPD

### Training the DPD - change to your modulation scheme of choice and re-run this cell


In [ ]:
M = 16
no_symbols= 100_000 # must be multiple of 5000 (seq_length)
nBits = int(no_symbols * np.log2(M))
SpSout = 2
mzmScale = 0.8
laserLinewidth = 100e3

dpd_model = build_model()
train_DPD(M, nBits, iteration_cnt=15, Data_Aided=True, SpSout=SpSout,mzmScale=mzmScale, CPR_Mode="bps", laserLinewidth=laserLinewidth) # here you can specify things like back_to_back enable, CPR mode, ..... etc but DONT change Data_Aided - for training this must be true


dpd_model.load_weights("best_model.weights.h5")

### Performance Evaluation / Testing

In [ ]:
# TODO: validate the accuracy of the postprocess func.

results_nodpd = []
results_dpd = []

# DA and CPR modes
modes_DataAided = [True, False]
modes_CPR = [ "bps"]

paramSymb = intialise_paramSymb(M, nBits, seed=333)

for da in modes_DataAided:
    for cpr_mode in modes_CPR:
            print("\n===================================================")
            print(f" Running:  M={M}, Data_Aided={da}, CPR_Mode={cpr_mode}")
            print("===================================================\n")
        
            # fresh symbol stream each run, ... do i need to change the seed?
            symbTx = symbolSource(paramSymb)


            ####################### W/O DPD #######################
            y_CPR_1, d, phaseEst = simulate_optical_system(symbTx, len(symbTx), M, 
                                                            Data_Aided=da, 
                                                            SpSout=SpSout,
                                                            CPR_Mode=cpr_mode, mzmScale=mzmScale, laserLinewidth=laserLinewidth)

            BER, SER, SNR, EVM, Q = perf_calc(symbTx, y_CPR_1, d, M, paramSymb)

            results_nodpd.append({
                "Modulation": M,
                "Data_Aided": da,
                "CPR": cpr_mode,
                "BER": BER,
                "SER": SER,
                "SNR": SNR,
                "EVM": EVM,
                "Qfactor": Q
            })

            ####################### W/ DPD #######################

            symbTx_nn = preprocess(symbTx)
            symbDPD = dpd_model.predict(symbTx_nn, verbose=0)
            symbDPD = merge_i_q(symbDPD).flatten()

            # run sys
            y_CPR_1, d, phaseEst = simulate_optical_system(symbDPD, len(symbDPD), M, 
                                                            Data_Aided=da, 
                                                            SpSout=SpSout,
                                                            CPR_Mode=cpr_mode, mzmScale=mzmScale, laserLinewidth=laserLinewidth
                                                            )

            # performance
            BER, SER, SNR, EVM, Q = perf_calc(symbTx, y_CPR_1, d, M, paramSymb)


            results_dpd.append({
                "Modulation": M,
                "Data_Aided": da,
                "CPR": cpr_mode,
                "BER": BER,
                "SER": SER,
                "SNR": SNR,
                "EVM": EVM,
                "Qfactor": Q
            })


In [ ]:
print("\n==================== W/O DPD Results ====================\n")
for r in results_nodpd:
    print(f"DA={r['Data_Aided']}, CPR={r['CPR']}: "
          f"BER={r['BER']:.2e}, SER={r['SER']:.2e}, "
          f"SNR={r['SNR']:.2f} dB, EVM={r['EVM']*100:.1f} %, Q={r['Qfactor']:.2f}")
    
print("\n==================== W/ DPD Results ====================\n")
for r in results_dpd:
    print(f"DA={r['Data_Aided']}, CPR={r['CPR']}: "
          f"BER={r['BER']:.2e}, SER={r['SER']:.2e}, "
          f"SNR={r['SNR']:.2f} dB, EVM={r['EVM']*100:.1f} %, Q={r['Qfactor']:.2f}")

In [ ]:
fir_i = dpd_model.get_layer('fir_i')
print("const float taps_I[NO_TAPS] = {")
for tap in fir_i.get_weights()[0].reshape(-1)[::-1]:
    print(f"    {tap},")
print("};")

fir_q = dpd_model.get_layer('fir_q')
print("const float taps_Q[NO_TAPS] = {")
for tap in fir_q.get_weights()[0].reshape(-1)[::-1]:
    print(f"    {tap},")
print("};")

#################
fcnn_1  = dpd_model.get_layer('dense')


print("const float weights_1[NEURONS_1*2] = {")
for weight in fcnn_1.get_weights()[0].reshape(-1):
    print(f"    {weight},")
print("};")

print("const float biases_1[NEURONS_1] = {")
for bias in fcnn_1.get_weights()[1].reshape(-1):
    print(f"    {bias},")
print("};")

################
fcnn_2  = dpd_model.get_layer('dense_1')


print("const float weights_2[NEURONS_2*NEURONS_1] = {")
for weight in fcnn_2.get_weights()[0].reshape(-1):
    print(f"    {weight},")
print("};")

print("const float biases_2[NEURONS_2] = {")
for bias in fcnn_2.get_weights()[1].reshape(-1):
    print(f"    {bias},")
print("};")

################
fcnn_3  = dpd_model.get_layer('dense_2')


print("const float weights_3[NEURONS_3*NEURONS_2] = {")
for weight in fcnn_3.get_weights()[0].reshape(-1):
    print(f"    {weight},")
print("};")

print("const float biases_3[NEURONS_3] = {")
for bias in fcnn_3.get_weights()[1].reshape(-1):
    print(f"    {bias},")
print("};")

In [ ]:
# FOR TESTBENCH

paramSymb = intialise_paramSymb(M, nBits, seed=333)
symbTx = symbolSource(paramSymb)

slice = preprocess(symbTx)[0]

x_to_hw = np.concatenate((slice[:,0], slice[:,1]), axis=0)

print("const float x[10000] = {")
for val in x_to_hw:    print(f"    {val},")
print("};")


In [79]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, Model, initializers, Sequential
tf.random.set_seed(333)
np.random.seed(333)
inputs = layers.Input(shape=(None, 1)) # 2 for I and Q
outputs = layers.Conv1D(1, 101, padding='same', use_bias=False)(inputs) # 100 taps was a sweet spot, 20-ish fails to converge, tiker with different values.

model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2), loss='mse') # ILA)

x = np.random.randn(500).reshape(1,-1)
y = model.predict(x)

y.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


(1, 500, 1)

In [83]:
print("const float x[NO_SYMBOLS] = {")
for val in x.reshape(-1):    print(f"    {val},")
print("};")

taps = model.get_weights()[0].reshape(-1)

print("const float taps[NO_TAPS] = {")
for val in taps:    print(f"    {val},")
print("};")


const float x[NO_SYMBOLS] = {
    1.7171842924792238,
    0.3246933317323878,
    -0.47454340095763164,
    -0.1379183025564171,
    0.5249395681621802,
    -0.35783583263955154,
    -0.7282049175161732,
    -0.6420864549339821,
    -0.3170476568930646,
    -1.219001429125632,
    -0.05198034222757668,
    0.6940273099274554,
    -0.12359775637870678,
    1.7736170919975704,
    0.27269253540080846,
    -0.21295821412076502,
    -0.47321243903774063,
    -0.3135010787602591,
    1.6216738837559117,
    0.7179492678379249,
    -2.1218129155586563,
    -0.24258443093039608,
    -0.8268112193921369,
    -1.2774569925993355,
    0.31806252961576004,
    0.2671719691485973,
    1.8255946884438583,
    0.1524720619664627,
    0.5205848195158618,
    -1.1216166102583114,
    2.1630675155558428,
    1.6948861075643609,
    -0.5851633690126306,
    -1.0224918167832164,
    0.4315552389895657,
    -1.1693672473389496,
    -0.7090900339311214,
    -0.411258046588551,
    -0.9869319041185098,
    

In [ ]:
## TF Output
for i, v in enumerate(y.reshape(-1)):
    print(f"    {i}: , {v},")


    0: , -0.37756484746932983,
    1: , 1.4098231792449951,
    2: , 0.5936123132705688,
    3: , 0.09132861346006393,
    4: , 0.20556877553462982,
    5: , 0.0740874782204628,
    6: , 1.009292721748352,
    7: , 0.46731019020080566,
    8: , 0.3061664402484894,
    9: , 0.23554271459579468,
    10: , -1.5124363899230957,
    11: , -2.063234329223633,
    12: , -0.6339417695999146,
    13: , -1.512237787246704,
    14: , -0.12303920835256577,
    15: , -0.1867632269859314,
    16: , -1.8140430450439453,
    17: , -0.7491092681884766,
    18: , -1.3804548978805542,
    19: , 0.8027524948120117,
    20: , -0.32721683382987976,
    21: , 1.7470933198928833,
    22: , 1.7674280405044556,
    23: , 0.7981891632080078,
    24: , -0.10214406996965408,
    25: , 0.6915494203567505,
    26: , 1.8792012929916382,
    27: , 0.908196210861206,
    28: , -0.08855356276035309,
    29: , 0.20535428822040558,
    30: , 0.5862257480621338,
    31: , -0.6260982751846313,
    32: , 0.43865424394607544,

HLS KErnel simu output:

breakpoint here
i: 0, sig_out: -0.377565
i: 1, sig_out: 1.409823
i: 2, sig_out: 0.739559
i: 3, sig_out: -0.393968
i: 4, sig_out: 0.571317
i: 5, sig_out: 0.403379
i: 6, sig_out: 0.449122
i: 7, sig_out: 0.656070
i: 8, sig_out: 0.071662
i: 9, sig_out: 0.094294
i: 10, sig_out: -1.227247
i: 11, sig_out: -1.359630
i: 12, sig_out: -1.379943
i: 13, sig_out: -1.227072
i: 14, sig_out: -0.255741
i: 15, sig_out: -0.788571
i: 16, sig_out: -0.832148
i: 17, sig_out: -0.851477
i: 18, sig_out: -0.693505
i: 19, sig_out: 0.132188
i: 20, sig_out: -0.295559
i: 21, sig_out: 0.791148
i: 22, sig_out: 2.036424
i: 23, sig_out: 1.253592
i: 24, sig_out: -0.420170
i: 25, sig_out: 1.234415
i: 26, sig_out: 0.960436
i: 27, sig_out: 1.421155
i: 28, sig_out: -0.856925
i: 29, sig_out: 1.022167
i: 30, sig_out: -0.400685
i: 31, sig_out: 0.626570
i: 32, sig_out: 0.166665
i: 33, sig_out: -0.443562
i: 34, sig_out: 1.183799
i: 35, sig_out: 0.833440
i: 36, sig_out: 1.340789
i: 37, sig_out: 1.469050
i: 38, sig_out: 1.127164
i: 39, sig_out: -0.538813
i: 40, sig_out: -0.897463
i: 41, sig_out: -1.163028
i: 42, sig_out: -0.355981
i: 43, sig_out: -1.311434
i: 44, sig_out: -0.848047
i: 45, sig_out: 0.438237
i: 46, sig_out: -0.926963
i: 47, sig_out: -0.610918
i: 48, sig_out: 0.480198
i: 49, sig_out: -0.158668
i: 50, sig_out: 0.236226
i: 51, sig_out: 0.421486
i: 52, sig_out: -0.203122
i: 53, sig_out: -0.816339
i: 54, sig_out: -0.307152
i: 55, sig_out: 0.343588
i: 56, sig_out: 1.264156
i: 57, sig_out: -0.373586
i: 58, sig_out: -0.463127
i: 59, sig_out: -1.425737
i: 60, sig_out: 0.782659
i: 61, sig_out: 0.711052
i: 62, sig_out: 1.390300
i: 63, sig_out: -0.606075
i: 64, sig_out: 0.053196
i: 65, sig_out: -0.884989
i: 66, sig_out: -0.442118
i: 67, sig_out: 0.536218
i: 68, sig_out: 0.517015
i: 69, sig_out: -0.493210
i: 70, sig_out: 0.046445
i: 71, sig_out: 0.296580
i: 72, sig_out: -0.297554
i: 73, sig_out: -0.036221
i: 74, sig_out: -0.468350
i: 75, sig_out: 1.762491
i: 76, sig_out: -0.408836
i: 77, sig_out: -0.742177
i: 78, sig_out: 0.710115
i: 79, sig_out: -1.653027
i: 80, sig_out: -1.793279
i: 81, sig_out: 0.123029
i: 82, sig_out: -1.586307
i: 83, sig_out: 0.004679
i: 84, sig_out: 0.302621
i: 85, sig_out: 0.571385
i: 86, sig_out: -0.525713
i: 87, sig_out: -0.973796
i: 88, sig_out: -0.112895
i: 89, sig_out: -1.339967
i: 90, sig_out: 1.432183
i: 91, sig_out: 0.814107
i: 92, sig_out: -0.733778
i: 93, sig_out: -1.897549
i: 94, sig_out: 0.078556
i: 95, sig_out: 0.150389
i: 96, sig_out: 0.514407
i: 97, sig_out: -0.745215
i: 98, sig_out: 1.534804
i: 99, sig_out: 0.441538
i: 100, sig_out: -0.581295
i: 101, sig_out: 0.538621
i: 102, sig_out: 0.779293
i: 103, sig_out: -1.077984
i: 104, sig_out: 1.172372
i: 105, sig_out: 0.892633
i: 106, sig_out: -0.374126
i: 107, sig_out: -1.762959
i: 108, sig_out: -1.062228
i: 109, sig_out: -0.957159
i: 110, sig_out: -1.152268
i: 111, sig_out: 0.763131
i: 112, sig_out: -0.706092
i: 113, sig_out: -0.849625
i: 114, sig_out: -0.309304
i: 115, sig_out: 0.749719
i: 116, sig_out: -0.010155
i: 117, sig_out: 0.254695
i: 118, sig_out: 0.281211
i: 119, sig_out: -0.278385
i: 120, sig_out: 1.197996
i: 121, sig_out: -0.272842
i: 122, sig_out: 0.418315
i: 123, sig_out: 0.327087
i: 124, sig_out: 1.252052
i: 125, sig_out: 2.332922
i: 126, sig_out: 0.197778
i: 127, sig_out: -0.271106
i: 128, sig_out: 2.229794
i: 129, sig_out: -0.475859
i: 130, sig_out: 1.326974
i: 131, sig_out: 0.453089
i: 132, sig_out: 0.435190
i: 133, sig_out: -1.086090
i: 134, sig_out: -0.049037
i: 135, sig_out: 0.273771
i: 136, sig_out: 0.411299
i: 137, sig_out: -0.420470
i: 138, sig_out: -0.172912
i: 139, sig_out: 1.816985
i: 140, sig_out: -0.544868
i: 141, sig_out: 2.052361
i: 142, sig_out: 0.118511
i: 143, sig_out: -0.676622
i: 144, sig_out: 0.950795
i: 145, sig_out: -0.686172
i: 146, sig_out: 0.019280
i: 147, sig_out: -0.395052
i: 148, sig_out: -1.054684
i: 149, sig_out: -0.299366
i: 150, sig_out: 0.270564
i: 151, sig_out: -1.099626
i: 152, sig_out: -0.019352
i: 153, sig_out: -0.599932
i: 154, sig_out: -1.193228
i: 155, sig_out: 1.442467
i: 156, sig_out: -2.691443
i: 157, sig_out: -0.809470
i: 158, sig_out: -1.421805
i: 159, sig_out: -1.316926
i: 160, sig_out: -0.518695
i: 161, sig_out: -2.000792
i: 162, sig_out: -0.494609
i: 163, sig_out: -0.363453
i: 164, sig_out: 0.829256
i: 165, sig_out: 1.235190
i: 166, sig_out: 1.537810
i: 167, sig_out: -0.885245
i: 168, sig_out: -0.124222
i: 169, sig_out: -0.436911
i: 170, sig_out: 1.567576
i: 171, sig_out: 0.811848
i: 172, sig_out: 0.375035
i: 173, sig_out: -0.714066
i: 174, sig_out: 0.367771
i: 175, sig_out: -0.923987
i: 176, sig_out: 1.058432
i: 177, sig_out: -0.526049
i: 178, sig_out: 0.003427
i: 179, sig_out: -0.004880
i: 180, sig_out: -0.246470
i: 181, sig_out: 0.105132
i: 182, sig_out: 0.021771
i: 183, sig_out: 0.458463
i: 184, sig_out: 1.501294
i: 185, sig_out: -0.231143
i: 186, sig_out: 0.300915
i: 187, sig_out: 1.037057
i: 188, sig_out: -1.335182
i: 189, sig_out: -0.719118
i: 190, sig_out: 0.208350
i: 191, sig_out: -0.592549
i: 192, sig_out: -0.049384
i: 193, sig_out: 0.053804
i: 194, sig_out: -1.095993
i: 195, sig_out: 1.005796
i: 196, sig_out: -1.080919
i: 197, sig_out: 1.430284
i: 198, sig_out: -1.264866
i: 199, sig_out: -0.299685
i: 200, sig_out: 1.194702
i: 201, sig_out: -0.390298
i: 202, sig_out: -0.280154
i: 203, sig_out: 0.537155
i: 204, sig_out: -0.129570
i: 205, sig_out: -0.469303
i: 206, sig_out: -0.269195
i: 207, sig_out: -0.436765
i: 208, sig_out: -0.152881
i: 209, sig_out: -1.093591
i: 210, sig_out: 0.309962
i: 211, sig_out: -0.908500
i: 212, sig_out: -1.719655
i: 213, sig_out: 0.120481
i: 214, sig_out: -0.823142
i: 215, sig_out: -1.482744
i: 216, sig_out: 2.140819
i: 217, sig_out: -0.018273
i: 218, sig_out: 0.811580
i: 219, sig_out: 0.100755
i: 220, sig_out: 1.295918
i: 221, sig_out: 0.030658
i: 222, sig_out: -0.152444
i: 223, sig_out: 0.157937
i: 224, sig_out: -0.101294
i: 225, sig_out: -0.767946
i: 226, sig_out: -0.615100
i: 227, sig_out: 1.222572
i: 228, sig_out: -1.485012
i: 229, sig_out: 0.146327
i: 230, sig_out: 0.332126
i: 231, sig_out: 1.911017
i: 232, sig_out: 0.620834
i: 233, sig_out: 1.739285
i: 234, sig_out: 0.144862
i: 235, sig_out: -1.221442
i: 236, sig_out: 1.175573
i: 237, sig_out: 1.056004
i: 238, sig_out: -0.061331
i: 239, sig_out: 0.356099
i: 240, sig_out: 1.383662
i: 241, sig_out: -1.222516
i: 242, sig_out: 0.729981
i: 243, sig_out: 0.004102
i: 244, sig_out: 0.639756
i: 245, sig_out: -0.370256
i: 246, sig_out: 2.270390
i: 247, sig_out: 1.767292
i: 248, sig_out: 0.765076
i: 249, sig_out: 0.187546
i: 250, sig_out: 0.260831
i: 251, sig_out: 1.049155
i: 252, sig_out: 0.672660
i: 253, sig_out: 1.832997
i: 254, sig_out: 0.505229
i: 255, sig_out: -1.497107
i: 256, sig_out: 0.415199
i: 257, sig_out: -0.159280
i: 258, sig_out: -0.432200
i: 259, sig_out: -0.520636
i: 260, sig_out: 1.308587
i: 261, sig_out: 0.131326
i: 262, sig_out: 0.320275
i: 263, sig_out: -1.489316
i: 264, sig_out: 0.623391
i: 265, sig_out: -0.605608
i: 266, sig_out: 1.304175
i: 267, sig_out: 1.009042
i: 268, sig_out: -0.099885
i: 269, sig_out: 0.907584
i: 270, sig_out: 0.124790
i: 271, sig_out: 0.747848
i: 272, sig_out: 0.671718
i: 273, sig_out: 1.898262
i: 274, sig_out: -0.724244
i: 275, sig_out: -0.021279
i: 276, sig_out: 0.400365
i: 277, sig_out: 0.880410
i: 278, sig_out: -0.447032
i: 279, sig_out: 0.418409
i: 280, sig_out: 0.468611
i: 281, sig_out: 0.798378
i: 282, sig_out: 0.551550
i: 283, sig_out: -0.194229
i: 284, sig_out: 0.391755
i: 285, sig_out: 0.506052
i: 286, sig_out: 0.966453
i: 287, sig_out: 0.549548
i: 288, sig_out: 0.124077
i: 289, sig_out: 0.807116
i: 290, sig_out: 0.709464
i: 291, sig_out: -0.198375
i: 292, sig_out: 0.177645
i: 293, sig_out: 0.952572
i: 294, sig_out: -1.126269
i: 295, sig_out: -0.738034
i: 296, sig_out: 1.128032
i: 297, sig_out: 0.546566
i: 298, sig_out: 0.373176
i: 299, sig_out: -0.508074
i: 300, sig_out: -0.471223
i: 301, sig_out: 0.410378
i: 302, sig_out: -0.507303
i: 303, sig_out: 0.227906
i: 304, sig_out: -2.333211
i: 305, sig_out: -0.107605
i: 306, sig_out: -0.552831
i: 307, sig_out: 0.239959
i: 308, sig_out: -1.100860
i: 309, sig_out: -1.196956
i: 310, sig_out: -0.963925
i: 311, sig_out: -0.377509
i: 312, sig_out: -1.512766
i: 313, sig_out: -1.445904
i: 314, sig_out: -0.364685
i: 315, sig_out: -1.383200
i: 316, sig_out: -1.559075
i: 317, sig_out: -0.748320
i: 318, sig_out: -0.220268
i: 319, sig_out: -1.806999
i: 320, sig_out: -1.151311
i: 321, sig_out: -0.821033
i: 322, sig_out: 0.269454
i: 323, sig_out: 0.609748
i: 324, sig_out: -1.877404
i: 325, sig_out: 0.107177
i: 326, sig_out: 0.281992
i: 327, sig_out: 0.181518
i: 328, sig_out: -0.329604
i: 329, sig_out: -1.197479
i: 330, sig_out: -0.917226
i: 331, sig_out: -0.018244
i: 332, sig_out: -0.299136
i: 333, sig_out: -0.762521
i: 334, sig_out: 0.215674
i: 335, sig_out: -0.862518
i: 336, sig_out: 0.868809
i: 337, sig_out: 0.187848
i: 338, sig_out: 0.184650
i: 339, sig_out: -1.116145
i: 340, sig_out: 0.661042
i: 341, sig_out: 1.467747
i: 342, sig_out: 0.630583
i: 343, sig_out: 0.992660
i: 344, sig_out: 0.332665
i: 345, sig_out: 0.513227
i: 346, sig_out: 0.687383
i: 347, sig_out: 2.379984
i: 348, sig_out: 0.563985
i: 349, sig_out: 1.703792
i: 350, sig_out: 0.789949
i: 351, sig_out: 1.970650
i: 352, sig_out: 0.276262
i: 353, sig_out: 0.466635
i: 354, sig_out: -0.112793
i: 355, sig_out: 0.499570
i: 356, sig_out: 1.599855
i: 357, sig_out: -0.008078
i: 358, sig_out: -0.946060
i: 359, sig_out: -0.070043
i: 360, sig_out: -0.133649
i: 361, sig_out: -0.498790
i: 362, sig_out: 0.086540
i: 363, sig_out: -0.720719
i: 364, sig_out: -0.384491
i: 365, sig_out: 0.091500
i: 366, sig_out: -0.824731
i: 367, sig_out: 1.546449
i: 368, sig_out: -1.129495
i: 369, sig_out: 1.400169
i: 370, sig_out: 0.513148
i: 371, sig_out: 1.238018
i: 372, sig_out: -0.796268
i: 373, sig_out: -0.273794
i: 374, sig_out: -0.327273
i: 375, sig_out: -1.314859
i: 376, sig_out: 0.593241
i: 377, sig_out: 0.485172
i: 378, sig_out: -0.913820
i: 379, sig_out: -0.365440
i: 380, sig_out: -0.287914
i: 381, sig_out: -1.595014
i: 382, sig_out: -0.167094
i: 383, sig_out: 0.343183
i: 384, sig_out: -0.825926
i: 385, sig_out: -0.970843
i: 386, sig_out: -0.773371
i: 387, sig_out: -0.467154
i: 388, sig_out: -0.457746
i: 389, sig_out: -0.669733
i: 390, sig_out: 0.373227
i: 391, sig_out: -1.466466
i: 392, sig_out: -0.803291
i: 393, sig_out: -1.392091
i: 394, sig_out: -0.730995
i: 395, sig_out: -1.371340
i: 396, sig_out: -1.599092
i: 397, sig_out: -1.124393
i: 398, sig_out: -0.202855
i: 399, sig_out: -1.394107
i: 400, sig_out: -0.493632
i: 401, sig_out: -0.778311
i: 402, sig_out: 0.354221
i: 403, sig_out: -0.263679
i: 404, sig_out: 1.517450
i: 405, sig_out: -0.284539
i: 406, sig_out: -0.057994
i: 407, sig_out: -0.293857
i: 408, sig_out: -0.402020
i: 409, sig_out: -0.749880
i: 410, sig_out: -0.016398
i: 411, sig_out: -0.944514
i: 412, sig_out: -0.575428
i: 413, sig_out: -0.098469
i: 414, sig_out: -0.221284
i: 415, sig_out: -1.301618
i: 416, sig_out: 0.431603
i: 417, sig_out: 1.068647
i: 418, sig_out: 0.175594
i: 419, sig_out: 1.145069
i: 420, sig_out: 1.331894
i: 421, sig_out: 0.623796
i: 422, sig_out: -0.065964
i: 423, sig_out: 2.051728
i: 424, sig_out: 0.967941
i: 425, sig_out: -0.085554
i: 426, sig_out: -0.544055
i: 427, sig_out: -0.103353
i: 428, sig_out: 0.163135
i: 429, sig_out: -0.016918
i: 430, sig_out: -0.324899
i: 431, sig_out: -0.536937
i: 432, sig_out: -0.913170
i: 433, sig_out: -0.903841
i: 434, sig_out: 1.140100
i: 435, sig_out: -0.141645
i: 436, sig_out: -0.218081
i: 437, sig_out: 1.188199
i: 438, sig_out: -1.210012
i: 439, sig_out: 0.495684
i: 440, sig_out: -0.405284
i: 441, sig_out: -0.046271
i: 442, sig_out: -0.872684
i: 443, sig_out: 0.149518
i: 444, sig_out: 0.438709
i: 445, sig_out: 0.639321
i: 446, sig_out: 0.948516
i: 447, sig_out: 0.117628
i: 448, sig_out: -0.767336
i: 449, sig_out: -2.142231
i: 450, sig_out: 0.000000
i: 451, sig_out: 0.000000
i: 452, sig_out: 0.000000
i: 453, sig_out: 0.000000
i: 454, sig_out: 0.000000
i: 455, sig_out: 0.000000
i: 456, sig_out: 0.000000
i: 457, sig_out: 0.000000
i: 458, sig_out: 0.000000
i: 459, sig_out: 0.000000
i: 460, sig_out: 0.000000
i: 461, sig_out: 0.000000
i: 462, sig_out: 0.000000
i: 463, sig_out: 0.000000
i: 464, sig_out: 0.000000
i: 465, sig_out: 0.000000
i: 466, sig_out: 0.000000
i: 467, sig_out: 0.000000
i: 468, sig_out: 0.000000
i: 469, sig_out: 0.000000
i: 470, sig_out: 0.000000
i: 471, sig_out: 0.000000
i: 472, sig_out: 0.000000
i: 473, sig_out: 0.000000
i: 474, sig_out: 0.000000
i: 475, sig_out: 0.000000
i: 476, sig_out: 0.000000
i: 477, sig_out: 0.000000
i: 478, sig_out: 0.000000
i: 479, sig_out: 0.000000
i: 480, sig_out: 0.000000
i: 481, sig_out: 0.000000
i: 482, sig_out: 0.000000
i: 483, sig_out: 0.000000
i: 484, sig_out: 0.000000
i: 485, sig_out: 0.000000
i: 486, sig_out: 0.000000
i: 487, sig_out: 0.000000
i: 488, sig_out: -1.#QNAN0
i: 489, sig_out: -1.#QNAN0
i: 490, sig_out: 0.000000
i: 491, sig_out: 0.000000
i: 492, sig_out: 0.000000
i: 493, sig_out: 0.000000
i: 494, sig_out: 0.000000
i: 495, sig_out: 0.000000
i: 496, sig_out: 0.000000
i: 497, sig_out: 0.000000
i: 498, sig_out: 0.000000
i: 499, sig_out: 0.000000